# Feature engineering ของ CENTRAL

อ่าน split จาก build_dataset.ipynb แล้วสร้างฟีเจอร์ 19 คอลัมน์สำหรับโมเดล โดยใช้ preprocessor ตัวเดียวกับทุก split


## 1. เตรียม path และ schema ฟีเจอร์

หาโฟลเดอร์ CENTRAL และกำหนดคอลัมน์ตัวเลข หมวดหมู่ และลำดับฟีเจอร์ 19 คอลัมน์


In [ ]:
from pathlib import Path

PIPELINE_DIR = next(
    candidate
    for ancestor in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (ancestor, ancestor / "data-pipeline")
    if (candidate / "CENTRAL" / "feature_contract.json").is_file()
)

import hashlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import json
import joblib

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


PREFIXES = ["proto", "service", "state"]


NAMES = [
    "log_duration",
    "log_bytes_forward",
    "log_bytes_backward",
    "signed_log_bytes_ratio",
] + [
    f"{prefix}_{value}"
    for prefix, values in zip(PREFIXES, CATEGORIES.values())
    for value in values
]


CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


def require(condition, message):
    if not condition:
        raise ValueError(message)


## 2. กำหนดการแปลงฟีเจอร์

กำหนด log transform, การ scale ค่าตัวเลข และ one-hot encoding ตาม schema กลาง


In [ ]:
def derived(f):
    x = np.log1p(f[NUMERIC].to_numpy(float))
    return np.column_stack([x, x[:, 1] - x[:, 2]])


def fit_preprocessor(f):
    enc = OneHotEncoder(
        categories=list(CATEGORIES.values()),
        sparse_output=False,
        handle_unknown="error",
    )
    enc.fit(f[list(CATEGORIES)])
    return {
        "scaler": StandardScaler().fit(derived(f)),
        "encoder": enc,
        "model_feature_names": NAMES,
        "fit_record_ids_sha256": hashlib.sha256(
            "\n".join(f.record_id.astype(str)).encode()
        ).hexdigest(),
        "fit_split": "train",
        "schema_version": 5,
    }


def transform_features(f, preprocessor):
    # Validation is shared with resampling; no silent clipping or imputation.
    for c in NUMERIC:
        require(np.isfinite(f[c]).all() and f[c].ge(0).all(), f"Invalid {c}")
    for c, values in CATEGORIES.items():
        require(f[c].isin(values).all(), f"Invalid {c}")
    x = np.column_stack(
        [
            preprocessor["scaler"].transform(derived(f)),
            preprocessor["encoder"].transform(f[list(CATEGORIES)]),
        ]
    )
    require(np.isfinite(x).all(), "Encoded features are not finite")
    result = pd.DataFrame(x, columns=NAMES)
    result.insert(0, "record_id", f.record_id.to_numpy())
    return result


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


## 3. เตรียมตัวอ่านและตรวจตาราง

ตรวจว่า features, labels และ metadata ตรงกัน และตรวจค่าที่จะป้อนให้ preprocessor


In [ ]:
def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


## 4. Fit และบันทึก preprocessor จาก train

อ่าน train ที่ผ่านการตรวจแล้ว ใช้ข้อมูลชุดนี้ fit scaler และ encoder แล้วบันทึก preprocessor.joblib


In [ ]:
central_dir = PIPELINE_DIR / "CENTRAL"

c=Path(central_dir);o=c/'output';out=o/'feature_engineering';out.mkdir(parents=True,exist_ok=True)
train=read_tables(o/'splits/train');validate_tables(*train)
pre=fit_preprocessor(train[0]);joblib.dump(pre,out/'preprocessor.joblib')


## 5. แปลงข้อมูลทั้งสี่ split

ใช้ preprocessor ที่ fit แล้วกับ train, train_full, validation และ test จากนั้นเขียนฟีเจอร์พร้อม labels และ metadata


In [ ]:
reports=[]
for split in ['train','train_full','validation','test']:
    f,l,m=read_tables(o/'splits'/split);validate_tables(f,l,m)
    dest=out/split;dest.mkdir(exist_ok=True)
    transform_features(f,pre).to_csv(dest/'features.csv',index=False)
    l.to_csv(dest/'labels.csv',index=False);m.to_csv(dest/'metadata.csv',index=False)
    reports.append({'split':split,'rows':len(f),'model_features':len(NAMES)})


## 6. บันทึก schema และรายงาน

บันทึกชื่อฟีเจอร์ตามลำดับ จำนวนแถวของแต่ละ split และ checksum ของรหัสแถวที่ใช้ fit


In [ ]:
pd.DataFrame(reports).to_csv(out/'feature_engineering_report.csv',index=False)
write_json(out/'feature_schema.json',{'schema_version':5,'model_feature_names':NAMES,'fit_split':'train','fit_record_ids_sha256':pre['fit_record_ids_sha256'],'numeric_scaling':'StandardScaler fitted on train','availability_policy':'other with false metadata flag is diagnostic-only'})
print(reports)
